# The Anatomy of Tool Calling: No Magic, Just Prompts
Many developers think "Tool Calling" is a separate neural network or a special mode. It is not. 

**The Truth:** Tool calling is just highly optimized **Structured Prompting**. 
When you provide a "Tool Definition" to an LLM:
1. The API appends your tool's JSON schema to the System Prompt.
2. The model is fine-tuned to recognize when a user query matches that schema.
3. Instead of answering the user, the model outputs a specific JSON block called a `tool_call`.

Today, we will inspect the raw payloads to see exactly how this "Handshake" works.

In [1]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)
client = OpenAI()

print("🔧 Tool Anatomy Environment Ready!")

🔧 Tool Anatomy Environment Ready!


## 1. The Blueprint: JSON Schema

To the LLM, a tool is just a description. We must tell it the name, what it does, and what parameters it needs. This is the **Blueprint**.

In [2]:
# This is NOT a function yet. It is just a DESCRIPTION of a function.
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "Get the current stock price for a given ticker symbol.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {
                        "type": "string",
                        "description": "The stock ticker symbol (e.g. AAPL, TSLA)"
                    }
                },
                "required": ["ticker"]
            }
        }
    }
]

print("📋 Tool Blueprint Defined.")

📋 Tool Blueprint Defined.


## 2. Triggering the Call: The Model's Decision

We send a query that *should* require the tool. We will inspect the raw response to see that the model **does not answer the question**—it only asks for the tool.

In [3]:
#query = "What is the current price of NVIDIA stock?"
query = "What is the currency of China?"

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": query}],
    tools=tools, 
    tool_choice="auto" # The model decides whether to use the tool
)

message = response.choices[0].message

print(f"User Query: {query}")
print("-" * 30)
print(f"Model Content: {message.content}") # This will likely be None!
print(f"Tool Calls: {json.dumps(message.tool_calls[0].function.model_dump(), indent=2) if message.tool_calls else 'None'}")

print("\n💡 ARCHITECT'S NOTE: Notice how 'content' is empty. The model has paused and is waiting for YOU to run the code.")

User Query: What is the currency of China?
------------------------------
Model Content: The currency of China is the Chinese Yuan, which is often denoted by the symbol "¥" and the currency code "CNY."
Tool Calls: None

💡 ARCHITECT'S NOTE: Notice how 'content' is empty. The model has paused and is waiting for YOU to run the code.


## 3. Proving No Magic: The "Manual" Tool Call

To prove that tool calling is just structured prompting, we can achieve the exact same result without the `tools` parameter, just by using a strict System Prompt.

In [4]:
system_prompt = """
You are a helpful assistant. 
If the user asks for a stock price, you must NOT answer directly. 
Instead, you must return a JSON object in this format: 
{"action": "get_stock_price", "ticker": "SYMBOL"}
"""

response_manual = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "What is the currency of China?"}
    ],
    response_format={ "type": "json_object" } # Forces JSON output
)

print("--- Manual 'Fake' Tool Call ---")
print(response_manual.choices[0].message.content)

print("\n🚀 Conclusion: Native 'Tool Calling' is just a cleaner, more reliable version of this exact pattern.")

--- Manual 'Fake' Tool Call ---
{"action": "get_currency", "country": "China"}

🚀 Conclusion: Native 'Tool Calling' is just a cleaner, more reliable version of this exact pattern.
